# Contamination Detection via Context (CoDeC) for *Alle mot 1*

This notebook applies **Contamination Detection via Context (CoDeC)** to the
Norwegian television game show *Alle mot 1*. It follows the method introduced
by Zawalski et al. (2026) in [Detecting Data Contamination in LLMs via In-Context
Learning](https://arxiv.org/abs/2510.27055), adapting the authors'
[reference implementation](https://github.com/NVIDIA-NeMo/Evaluator) to a
hosted Fireworks model.

This notebook is configured to use
`accounts/fireworks/models/gpt-oss-20b`. The model identifier documents
the exact configuration.


## Problem definition

Given a language model $M$ and a candidate dataset $\mathcal{D} = \{x_i\}_{i=1}^{N}$, where each $x_i$ is a text sequence, the objective is to quantify contamination. That is, whether the model was exposed to the dataset or closely related data during training, and therefore relies on memorization.


## Key Idea

CoDeC is based on the observation that language models respond differently to same-dataset context depending on whether they have encountered that data before.

For an **unseen dataset**, a context example can reveal useful information about its style, structure, or vocabulary, which usually increases the model's confidence in the target text. For a **contaminated dataset**, the model may already have internalized these patterns, so the context adds little new information and may even disrupt memorized patterns, causing confidence to decrease.

Therefore:
- **Higher confidence without context** → Likely contaminated
- **Higher confidence with context** → Likely not contaminated


## Method

CoDeC compares the model's average token log probability for the same target
sample under two conditions:

1. **Baseline:** the target sample is evaluated on its own.
2. **With context:** the target sample is evaluated again, now with another sample from the same dataset placed before it.

The model is not asked to generate any output. Instead, it scores the tokens present in the target input sequence by predicting each token from the tokens
before it.


## Pipeline

For each target sample $x$ in dataset $\mathcal{D}$:

1. Calculate the average log probability of the target tokens without context.
2. Select another sample from $\mathcal{D} \setminus \{x\}$, place it before
   the target, and calculate the target-token log probabilities again. The
   context-token probabilities are not included.
3. Measure the change in confidence for context draw $r$:

   $$\Delta_r(x) = L_{\text{context},r}(x) - L_{\text{baseline}}(x)$$

4. Repeat the comparison over five context draws and calculate the mean change:

   $$\overline{\Delta}(x) = \frac{1}{5}\sum_{r=1}^{5}\Delta_r(x)$$

   Count the target as a contamination signal when
   $\overline{\Delta}(x) < 0$, meaning that the added context reduced the
   model's confidence on average.

The final CoDeC score is the percentage of target samples that produce a
negative mean confidence change:

$$
S_{\text{CoDeC}}(\mathcal{D}) =
\frac{100}{N}
\sum_{i=1}^{N}
\mathbb{1}\left[\overline{\Delta}(x_i) < 0\right]
$$


### Application to *Alle mot 1*

- Each **season** is treated as a separate dataset, so context questions are
  sampled only from the target question's season.
- Each **question transcript** is one naturally bounded sample. When working
  with continuous books or articles, the CoDeC paper divided the text into
  600-character chunks; the question transcripts are kept intact because they
  already form meaningful units with a median of 1,092 characters.
- The authors found that approximately 100 samples produced stable estimates. Our dataset contains 526 questions across seven seasons, with
  70–88 questions per season.
- Contamination is tested both within each season and in the dataset as a whole.

## Setup

In [ ]:
%pip install -q supabase fireworks-ai

import os
import random
import time
from typing import Any, Dict, List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from fireworks.client import Fireworks
from supabase import create_client


## Configuration



In [ ]:
model_id = "accounts/fireworks/models/gpt-oss-20b"
questions_table = "codec_questions"

# Each target question is compared by adding one context example.
# Since the context is sampled randomly, the value of Δ(x) is subject to some variance.
# To improve stability, we average the Δ(x) values over 5 seeds.
num_context_examples = 1
n_seeds = 5
seed = 42
token_range = (10, -1)
request_sleep = 0.7

supabase_url = os.environ.get("SUPABASE_URL")
supabase_key = os.environ.get("SUPABASE_KEY")
fireworks_api_key = os.environ.get("FIREWORKS_API_KEY")

if not supabase_url or not supabase_key or not fireworks_api_key:
    raise ValueError("Missing Supabase or Fireworks environment variables.")

db = create_client(supabase_url, supabase_key)


## Load Data
The query loads question transcripts and their identifying information from the database. Questions are ordered hierarchically by season, episode within season, and question number within episode.


In [ ]:
# The dataset contains fewer than Supabase's 1,000-row response limit.
columns = ["question_id", "season", "episode", "question_num", "transcript"]
fetch = (
    db.table(questions_table)
    .select(",".join(columns))
    .eq("include_for_codec", True)
    .eq("review_status", "READY")
    .order("season")
    .order("episode")
    .order("question_num")
    .limit(1000)
)
questions = pd.DataFrame(fetch.execute().data or [], columns=columns)

if questions[["question_id", "season", "transcript"]].isna().any().any():
    raise ValueError("Required question fields contain missing values.")

dataset_summary = (
    questions.assign(characters=questions["transcript"].str.len())
    .groupby("season", as_index=False)
    .agg(
        n_questions=("question_id", "size"),
        median_characters=("characters", "median"),
    )
)

print(f"Questions: {len(questions):,}")
display(dataset_summary)


## Core Implementation of CoDeC

The analysis consists of three components:

1. **Model handler:** Obtains tokens and their log probabilities from the language model.
2. **CoDeC detector:** Calculates the change in the target question's average log probability after adding context.
3. **Contamination detection pipeline:** Applies the CoDeC calculation to every question using five randomly sampled contexts and summarizes the results.

In [ ]:
class ModelHandler:
    """Handles model API calls and returns prompt token logprobs.

    Methods:
    get_logprobs_and_tokens: Sends text to Fireworks and requests its prompt tokens and log probs.
    _extract: Extracts and processes the tokens and log probs returned by Fireworks.
    """

    def __init__(
        self,
        model_name: str,
        api_key: str,
        verbose: bool = False,
    ):
        self.model_name = model_name
        self.client = Fireworks(api_key=api_key)
        self.verbose = verbose

        if self.verbose:
            print(f"Model ready: {model_name}")

    def get_logprobs_and_tokens(
        self,
        text: str,
    ) -> tuple[np.ndarray, List[str]]:
        """
        Get prompt logprobs and tokens for input text.

        Args:
            text: Text for which prompt logprobs are requested.

        Returns:
            Tuple containing the logprob array and matching token strings.
        """

        # First attempt: Ask Fireworks to evaluate the supplied prompt without generating text.
        # If succeeds, returns results and ends.
        try:
            response = self.client.completions.create(
                model=self.model_name,
                prompt=text,
                max_tokens=0,
                logprobs=1,
                echo=True,
            )
            return self._extract(response)

        except Exception as error:
            message = str(error).lower()
            if "rate_limit" in message or "rate limit" in message or "429" in message:
                raise

            # Second attempt: Some endpoints require one generated token.
            # Remove that token afterwards.
            response = self.client.completions.create(
                model=self.model_name,
                prompt=text,
                max_tokens=1,
                logprobs=1,
                echo=True,
            )
            return self._extract(response, drop_last=True)


    def _extract(
        self,
        response: Any,
        drop_last: bool = False,
    ) -> tuple[np.ndarray, List[str]]:
        """
        Extract prompt logprobs and tokens from a Fireworks response.

        Args:
            response: Response returned by the Fireworks completion API.
            drop_last: Whether to remove one generated token from the response.

        Returns:
            Tuple containing the logprob array and matching token strings.
        """
        # First level: Retrieve container holding all logprob-related information.
        # Fireworks places each returned completion in the choices list.
        completion = response.choices[0]
        logprobs_object = getattr(completion, "logprobs", None)
        if logprobs_object is None and isinstance(completion, dict):
            logprobs_object = completion.get("logprobs")
        if logprobs_object is None:
            raise ValueError("The response contains no prompt logprobs.")

        # Second level: Retrieve numerical logprobs.
        token_logprobs = getattr(logprobs_object, "token_logprobs", None)
        if token_logprobs is None and isinstance(logprobs_object, dict):
            token_logprobs = logprobs_object.get("token_logprobs")

        # Third level: Retrieve tokens
        tokens = getattr(logprobs_object, "tokens", None)
        if tokens is None and isinstance(logprobs_object, dict):
            tokens = logprobs_object.get("tokens")

        if token_logprobs is None or tokens is None:
            raise ValueError("Prompt tokens or token logprobs are unavailable.")

        logprobs = np.array(
            [np.nan if value is None else float(value) for value in token_logprobs],
            dtype=float,
        )
        tokens = list(tokens)

        if drop_last:
            logprobs = logprobs[:-1]
            tokens = tokens[:-1]

        return logprobs, tokens


In [ ]:
model_handler = ModelHandler(
    model_name=model_id,
    api_key=fireworks_api_key,
    verbose=True,
)


In [ ]:
class CoDeC:
    """Calculates CoDeC score by comparing target text with and without added context."""

    def __init__(self, token_range: tuple = token_range):
        """
        Initialize the contamination detector.

        Args:
            token_range: Range of target tokens to consider.
        """
        self.token_range = token_range

    def detect_contamination(
        self,
        target_text: str,
        context_examples: List[str],
        model_handler: ModelHandler,
    ) -> float | None:
        """
        Detect contamination for a single target text and context draw.

        Args:
            target_text: Text sample to test for contamination.
            context_examples: Other samples from the same dataset used as context.
            model_handler: Model handler used to obtain prompt logprobs.

        Returns:
            CoDeC delta (context minus baseline). A negative value is a
            contamination signal.
        """
        # Get target log probabilities without context.
        baseline_logprobs, baseline_tokens = (
            model_handler.get_logprobs_and_tokens(target_text)
        )
        target_length = len(baseline_logprobs)
        if target_length == 0:
            return None

        # Create context by joining examples.
        if not context_examples:
            raise ValueError("At least one context example is required.")
        context = "\n\n".join(context_examples)
        prompt_with_context = context + "\n\n" + target_text

        # Get target log probabilities after adding context.
        prompt_logprobs, prompt_tokens = (
            model_handler.get_logprobs_and_tokens(prompt_with_context)
        )
        if len(prompt_logprobs) < target_length:
            return None

        # The target appears at the end, so take the final target tokens.
        target_logprobs_after_context = prompt_logprobs[-target_length:]
        target_tokens_after_context = prompt_tokens[-target_length:]

        # Confirm that the same target tokens are compared in both conditions.
        if baseline_tokens != target_tokens_after_context:
            raise ValueError(
                "The target was not divided into the same tokens before "
                "and after adding context."
            )

        # Calculate average confidence for the specified token range.
        start_index, end_index = self.token_range
        if end_index == -1:
            end_index = target_length - 1
        else:
            end_index = min(end_index, target_length)

        if start_index >= end_index:
            print(f"Target text is too short: {target_text}")
            return None

        baseline = baseline_logprobs[start_index:end_index]
        after_context = target_logprobs_after_context[start_index:end_index]

        # Average the same finite target-token positions in both conditions.
        finite = np.isfinite(baseline) & np.isfinite(after_context)
        if not finite.any():
            return None

        confidence_baseline = np.mean(baseline[finite])
        confidence_after_context = np.mean(after_context[finite])

        # A negative difference means the target was easier without context.
        confidence_diff = confidence_after_context - confidence_baseline
        return float(confidence_diff)


In [ ]:
def contamination_detection_pipeline(
    model_handler: ModelHandler,
    dataset: pd.DataFrame,
    num_context_examples: int = 1,
    n_seeds: int = 5,
    seed: int = 42,
) -> Dict[str, Any]:
    """
    Run contamination detection on all season datasets.

    Args:
        model_handler: Model handler used for logprob inference.
        dataset: Dataframe with one row per question.
        num_context_examples: Number of same-season context questions.
        n_seeds: Number of random context draws per question.
        seed: Base random seed used to reproduce context selection.

    Returns:
        Dictionary with pooled, season, question, and context-draw results.
    """
    detector = CoDeC()
    rows = dataset.reset_index(drop=True).copy()


    rows["season_target_index"] = rows.groupby("season").cumcount()

    question_results = []
    seed_results = []
    print(
        f"Processing {len(rows)} questions across "
        f"{rows['season'].nunique()} seasons..."
    )

    # Process each question.
    for target_index in range(len(rows)):
        question_id = rows.loc[target_index, "question_id"]
        target_text = rows.loc[target_index, "transcript"]
        season = int(rows.loc[target_index, "season"])
        season_target_index = int(rows.loc[target_index, "season_target_index"])

        # Select context questions from the same season, excluding the target question.
        available_examples = rows.index[
            rows["season"].eq(season) & rows.index.to_series().ne(target_index)
        ].tolist()
        if not available_examples:
            raise ValueError(f"{question_id}: no same-season contexts are available.")

        deltas = []

        # Repeat with independently sampled contexts.
        for seed_index in range(n_seeds):
            random_seed = seed + season_target_index * 1000 + seed_index
            random_generator = random.Random(random_seed)

            if len(available_examples) >= num_context_examples:
                context_indices = random_generator.sample(
                    available_examples,
                    num_context_examples,
                )
            else:
                context_indices = available_examples

            context_examples = (
                rows.loc[context_indices, "transcript"].astype(str).tolist()
            )
            context_ids = (
                rows.loc[context_indices, "question_id"].astype(str).tolist()
            )

            delta = detector.detect_contamination(
                target_text,
                context_examples,
                model_handler,
            )
            time.sleep(request_sleep)

            if delta is not None and np.isfinite(delta):
                deltas.append(delta)
                seed_results.append({
                    "season": season,
                    "question_id": question_id,
                    "seed_index": seed_index,
                    "random_seed_used": random_seed,
                    "context_question_ids": "|".join(context_ids),
                    "delta": delta,
                })

        if len(deltas) != n_seeds:
            raise ValueError(
                f"{question_id}: expected {n_seeds} valid context draws, "
                f"but obtained {len(deltas)}."
            )

        # Average context draws before classifying the question.
        mean_delta = float(np.mean(deltas))
        question_results.append({
            "season": season,
            "question_id": question_id,
            "mean_delta": mean_delta,
            "delta_std": float(np.std(deltas, ddof=0)),
            "is_contaminated": mean_delta < 0,
            "n_seed_evals": len(deltas),
        })

    question_results = pd.DataFrame(question_results)
    seed_results = pd.DataFrame(seed_results)

    # Calculate one result per season.
    season_results = (
        question_results.groupby("season", as_index=False)
        .agg(
            n_questions=("question_id", "size"),
            codec_percent=("is_contaminated", lambda values: 100 * values.mean()),
            mean_delta=("mean_delta", "mean"),
            std_delta=("mean_delta", lambda values: values.std(ddof=0)),
        )
    )

    # Calculate one pooled result for the complete show.
    pooled_result = pd.DataFrame([{
        "season": "All seasons",
        "n_questions": len(question_results),
        "codec_percent": 100 * question_results["is_contaminated"].mean(),
        "mean_delta": question_results["mean_delta"].mean(),
        "std_delta": question_results["mean_delta"].std(ddof=0),
    }])
    summary_results = pd.concat(
        [pooled_result, season_results],
        ignore_index=True,
    )

    return {
        "summary_results": summary_results,
        "question_results": question_results,
        "seed_results": seed_results,
    }


## Run CoDeC

The pipeline performs five baseline/context comparisons per question, averages
the five deltas, and reports the pooled and season-level CoDeC scores.


In [ ]:
results = contamination_detection_pipeline(
    model_handler=model_handler,
    dataset=questions,
    num_context_examples=num_context_examples,
    n_seeds=n_seeds,
    seed=seed,
)

summary_results = results["summary_results"]
question_results = results["question_results"]
seed_results = results["seed_results"]

display(summary_results)


## Results

The left panel shows the distribution of five-draw mean deltas within each
season. The right panel pools the same question-level estimates across all
seasons. The red line marks the CoDeC decision threshold at zero.


In [ ]:
seasons = sorted(question_results["season"].unique())
season_deltas = [
    question_results.loc[
        question_results["season"].eq(season), "mean_delta"
    ].to_numpy()
    for season in seasons
]
pooled_deltas = question_results["mean_delta"].to_numpy()

fig, (ax_seasons, ax_pooled) = plt.subplots(
    1,
    2,
    figsize=(11, 4.5),
    gridspec_kw={"width_ratios": [2, 1]},
)

# Question-level delta distributions for each season.
positions = np.arange(1, len(seasons) + 1)
ax_seasons.boxplot(
    season_deltas,
    positions=positions,
    widths=0.55,
    showfliers=False,
)
jitter = np.random.default_rng(42)
for position, values in zip(positions, season_deltas):
    ax_seasons.scatter(
        jitter.normal(position, 0.05, len(values)),
        values,
        s=12,
        alpha=0.45,
    )
ax_seasons.axhline(0, color="#A23B3B", linewidth=1.2)
ax_seasons.set_xticks(positions, seasons)
ax_seasons.set_xlabel("Season")
ax_seasons.set_ylabel("Question-level mean delta")
ax_seasons.set_title("By season")

# Pooled distribution across all seasons.
ax_pooled.hist(pooled_deltas, bins=20, color="#6B8E7B", alpha=0.8)
ax_pooled.axvline(0, color="#A23B3B", linewidth=1.2)
ax_pooled.set_xlabel("Question-level mean delta")
ax_pooled.set_ylabel("Questions")
ax_pooled.set_title("All seasons")

pooled_score = 100 * question_results["is_contaminated"].mean()
ax_pooled.text(
    0.97,
    0.95,
    f"CoDeC: {pooled_score:.1f}%\nn = {len(question_results)}",
    transform=ax_pooled.transAxes,
    ha="right",
    va="top",
)

for axis in (ax_seasons, ax_pooled):
    axis.spines[["top", "right"]].set_visible(False)

fig.suptitle(f"CoDeC results: {model_id.split('/')[-1]}")
fig.tight_layout()
plt.show()
